In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import geopandas as gpd
from shapely.geometry import Point, Polygon, shape
import folium
import matplotlib.colors as mcolors
import datetime as dt
import numpy as np
import rasterio
from rasterio.features import shapes
from scipy.ndimage import label
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import rasterio as rio


In [3]:
folder_project = 'E:/'
region = 'aac'

start_year = 1988
end_year = 1996

for year in range(start_year, end_year+1):


    ds = xr.open_dataset(os.path.join(folder_project,'ROS_NC', 'ros_%s_%s.nc'%(year,region)))

    ros_2 = ds['ROS_2']

    # Load your DataArray 
    data_array = ros_2
    eps = 0.05
    min_samples = 20

    def apply_clustering(data_slice):
        coords = np.column_stack(np.where(data_slice == 1))
        
        if coords.shape[0] == 0:
            return data_slice
        
        scaler = StandardScaler()
        coords_scaled = scaler.fit_transform(coords)
        
        db = DBSCAN(eps=eps, min_samples=min_samples).fit(coords_scaled)
        labels = db.labels_
        
        mask = labels == -1
        
        for i, (x, y) in enumerate(coords):
            if mask[i]:
                data_slice[x, y] = 0
                
        return data_slice



    # Initialize a new DataArray for clustered results
    clustered_data_array = xr.DataArray(
        np.zeros_like(data_array.values),
        dims=data_array.dims,
        coords=data_array.coords
    )

    # Iterate over each time and orbit
    for orbit in data_array.orbit.values:
        for time in data_array.time.values:
            # Extract the slice for the specific time and orbit
            data_slice = data_array.sel(orbit=orbit, time=time).values
            
            # Apply clustering
            updated_slice = apply_clustering(data_slice)
                
            
            # Update the DataArray slice
            # Ensure proper assignment to the DataArray
            clustered_data_array.loc[dict(orbit=orbit, time=time)] = xr.DataArray(updated_slice, dims=('y', 'x'))

    # Save the clustered DataArray if needed
    clustered_data_array.to_netcdf(os.path.join(folder_project,'ROS_NC', 'ros_%s_%s_cluster_dbscan_%s_%s_grp_moins2.nc'%(str(year),region, str(eps).replace('.','_'), str(min_samples))))
    #clustered_data_array.to_netcdf(os.path.join(folder_project,'ROS_NC', 'ros_%s_%s_cluster_dbscan_%s_%s_grp_moins2_without_snowcon.nc'%(str(year),region, str(eps).replace('.','_'), str(min_samples))))

Test for one slice

In [ ]:
folder_project = 'E:/'
region = 'aac'
year = 2010
ds = xr.open_dataset(os.path.join(folder_project,'ROS_NC', 'ros_%s_%s.nc'%(year,region)))
#ds = xr.open_dataset(os.path.join(folder_project,'ROS_NC', 'ros_%s_%s_withoutsnowcon.nc'%(year,region)))

ros = ds['ROS']
#ims = ds['IMS_Surface_Values']
grp = ds['GRP']

ds['ROS_snow_ice_2'] = xr.where((ds['IMS_Surface_Values'].isin([3,4,364,365])) & (ds['GRP'] < -2), 1, 0)
#ds['ROS_snow_ice_2'] = xr.where((ds['GRP'] < -2), 1, 0)

ros_snow_ice_2 = ds['ROS_snow_ice_2']

In [ ]:
data_array = ros_snow_ice_2
eps = 0.05
min_samples = 50

clustered_data_array = xr.DataArray(
    np.zeros_like(data_array.values),
    dims=data_array.dims,
    coords=data_array.coords
)

#set time as 2010-10-01
time = dt.datetime(year, 10, 15)

orbit = 'M'


# Extract the slice for the specific time and orbit
data_slice = data_array.sel(orbit=orbit, time=time).values

# Apply clustering
coords = np.column_stack(np.where(data_slice == 1))

if coords.shape[0] == 0:
    updated_slice =  data_slice

else:
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)
    print(coords_scaled)

    db = DBSCAN(eps=eps, min_samples=min_samples).fit(coords_scaled)
    labels = db.labels_
    mask = labels == -1
    print('noise pixels detected: ', mask.sum())
    print('clustered pixels detected ', (~mask).sum())
    
    for i, (x, y) in enumerate(coords):
        if mask[i]:
            data_slice[x, y] = 0

#plot data_slice
plt.imshow(data_slice)

#save the plot
plt.savefig('E:/figures/banks/maps/clustered_data_array_%s.png'%(time.strftime('%Y%m%d')))




Test for a year

In [ ]:

# Load your DataArray 
data_array = ros_snow_ice_2
eps = 0.05
min_samples = 20

def apply_clustering(data_slice):
    coords = np.column_stack(np.where(data_slice == 1))
    
    if coords.shape[0] == 0:
        return data_slice
    
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)
    
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(coords_scaled)
    labels = db.labels_
    
    mask = labels == -1
    
    for i, (x, y) in enumerate(coords):
        if mask[i]:
            data_slice[x, y] = 0
            
    return data_slice



# Initialize a new DataArray for clustered results
clustered_data_array = xr.DataArray(
    np.zeros_like(data_array.values),
    dims=data_array.dims,
    coords=data_array.coords
)

# Iterate over each time and orbit
for orbit in data_array.orbit.values:
    for time in data_array.time.values:
        # Extract the slice for the specific time and orbit
        data_slice = data_array.sel(orbit=orbit, time=time).values
        
        # Apply clustering
        updated_slice = apply_clustering(data_slice)
               
        
        # Update the DataArray slice
        # Ensure proper assignment to the DataArray
        clustered_data_array.loc[dict(orbit=orbit, time=time)] = xr.DataArray(updated_slice, dims=('y', 'x'))


# Save the clustered DataArray if needed
# clustered_data_array.to_netcdf('clustered_file.nc')


In [ ]:
plt.imshow(data_slice)

In [ ]:
clustered_data_array

In [ ]:
# Save the clustered DataArray if needed
clustered_data_array.to_netcdf(os.path.join(folder_project,'ROS_NC', 'ros_%s_%s_cluster_dbscan_%s_%s_grp_moins2.nc'%(str(year),region, str(eps).replace('.','_'), str(min_samples))))

In [ ]:
region_extent = [-1030000,-600000,2600000,2990000]

In [ ]:
import rasterio as rio
#set time as 2010-10-01
time = dt.datetime(year, 10, 1)

orbit = 'M'

data_to_plot = clustered_data_array.sel(time=time, orbit=orbit)
# Sum before reprojection
sum_before = data_to_plot.sum().item()
print("Sum before reprojection:", sum_before)

# Reproject with nearest-neighbor resampling and increased resolution
data_to_plot_reprojected = data_to_plot.rio.reproject(
    "EPSG:3978", 
    resampling=rio.enums.Resampling.nearest,
    resolution=10000  # Example of increasing resolution
)

# Sum after reprojection
sum_after = data_to_plot_reprojected.sum().item()
print("Sum after reprojection:", sum_after)

In [ ]:
import rasterio as rio
#set time as 2010-10-01
time = dt.datetime(year, 10, 3)

orbit = 'M'

data_to_plot = clustered_data_array.sel(time=time, orbit=orbit)
print(data_to_plot.sum())

data_to_plot = data_to_plot.rio.write_crs("EPSG:6931")
# Reproject with nearest-neighbor resampling and increased resolution
data_to_plot = data_to_plot.rio.reproject(
    "EPSG:3978", 
    resampling=rio.enums.Resampling.nearest
)

data_to_plot = data_to_plot.where(data_to_plot == 1)

#drop nan values
data_to_plot = data_to_plot.dropna('x', how='all')
print(data_to_plot.sum())

plt.figure(figsize=(10, 8))

# Set up the map with the correct projection (EPSG:6931)
ax = plt.axes(projection=ccrs.epsg(3978))

# # Set the extent using the appropriate coordinates and CRS
ax.set_extent(region_extent, crs=ccrs.epsg(3978))

# Plot the data
data_to_plot.plot(ax=ax)

# Add coastlines or other map features
ax.coastlines()

plt.show()

In [ ]:
#set time as 2010-10-01
time = dt.datetime(year, 10, 3)

orbit = 'M'

data_to_plot = ros_snow_ice_2.sel(time=time, orbit=orbit)

data_to_plot = data_to_plot.rio.write_crs("EPSG:6931")
data_to_plot = data_to_plot.rio.reproject("EPSG:3978")

data_to_plot = data_to_plot.where(data_to_plot == 1)

#drop nan values
data_to_plot = data_to_plot.dropna('x', how='all')

plt.figure(figsize=(10, 8))

# Set up the map with the correct projection (EPSG:6931)
ax = plt.axes(projection=ccrs.epsg(3978))

# # Set the extent using the appropriate coordinates and CRS
ax.set_extent(region_extent, crs=ccrs.epsg(3978))

# Plot the data
data_to_plot.plot(ax=ax)

# Add coastlines or other map features
ax.coastlines()

plt.show()



# Testing for one time and orbit

In [ ]:
#set time as 2010-10-01
time = dt.datetime(year, 10, 3)

orbit = 'M'

ros_slice = ros.sel(time=time, orbit = orbit)

x_coords, y_coords = np.meshgrid(ros_slice['x'].values, ros_slice['y'].values)

grp_slice = grp.sel(time=time, orbit = orbit)

ros_snow_ice_10_slice = ros_snow_ice_10.sel(time=time, orbit = orbit)
ros_snow_ice_5_slice = ros_snow_ice_5.sel(time=time, orbit = orbit)
ros_snow_ice_2_slice = ros_snow_ice_2.sel(time=time, orbit = orbit)



In [ ]:


# Get coordinates of detection pixels
coords = np.column_stack(np.where(ros_snow_ice_2_slice == 1))

# Standardize the coordinates
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

# Apply DBSCAN
db = DBSCAN(eps=0.25, min_samples=100).fit(coords_scaled)
labels = db.labels_

# Keep only pixels that belong to a cluster
for i, (x, y) in enumerate(coords):
    if labels[i] == -1:
        ros_snow_ice_2_slice[x, y] = 0

In [ ]:
# Assign the CRS to your data if not already assigned
ros_snow_ice_10_slice = ros_snow_ice_10_slice.rio.write_crs("EPSG:6931")
ros_snow_ice_5_slice = ros_snow_ice_5_slice.rio.write_crs("EPSG:6931")
ros_snow_ice_2_slice = ros_snow_ice_2_slice.rio.write_crs("EPSG:6931")

# Reproject to EPSG:3978
ros_snow_ice_10_slice_reprojected = ros_snow_ice_10_slice.rio.reproject("EPSG:3978")
ros_snow_ice_5_slice_reprojected = ros_snow_ice_5_slice.rio.reproject("EPSG:3978")
ros_snow_ice_2_slice_reprojected = ros_snow_ice_2_slice.rio.reproject("EPSG:3978")

#drop nan values
ros_snow_ice_10_slice_reprojected = ros_snow_ice_10_slice_reprojected.dropna('x', how='all')
ros_snow_ice_5_slice_reprojected = ros_snow_ice_5_slice_reprojected.dropna('x', how='all')
ros_snow_ice_2_slice_reprojected = ros_snow_ice_2_slice_reprojected.dropna('x', how='all')

#keep only value of 1 
ros_snow_ice_10_slice_reprojected = ros_snow_ice_10_slice_reprojected.where(ros_snow_ice_10_slice_reprojected == 1)
ros_snow_ice_5_slice_reprojected = ros_snow_ice_5_slice_reprojected.where(ros_snow_ice_5_slice_reprojected == 1)
ros_snow_ice_2_slice_reprojected = ros_snow_ice_2_slice_reprojected.where(ros_snow_ice_2_slice_reprojected == 1)



plt.figure(figsize=(10, 8))

# Set up the map with the correct projection (EPSG:6931)
ax = plt.axes(projection=ccrs.epsg(3978))

# Set the extent using the appropriate coordinates and CRS
ax.set_extent([ros_snow_ice_10_slice_reprojected['x'].min(), ros_snow_ice_10_slice_reprojected['x'].max(), 
               ros_snow_ice_10_slice_reprojected['y'].min(), ros_snow_ice_10_slice_reprojected['y'].max()], crs=ccrs.epsg(3978))

# Plot the data
ros_snow_ice_2_slice_reprojected.plot(ax=ax)

# Add coastlines or other map features
ax.coastlines()

plt.show()




In [ ]:

plt.figure(figsize=(10, 8))

# Set up the map with the correct projection (EPSG:6931)
ax = plt.axes(projection=ccrs.epsg(3978))

# Set the extent using the appropriate coordinates and CRS
ax.set_extent([ros_snow_ice_10_slice_reprojected['x'].min(), ros_snow_ice_10_slice_reprojected['x'].max(), 
               ros_snow_ice_10_slice_reprojected['y'].min(), ros_snow_ice_10_slice_reprojected['y'].max()], crs=ccrs.epsg(3978))

# Plot the data
ros_snow_ice_5_slice_reprojected.plot(ax=ax)

# Add coastlines or other map features
ax.coastlines()

plt.show()


In [ ]:

plt.figure(figsize=(10, 8))

# Set up the map with the correct projection (EPSG:6931)
ax = plt.axes(projection=ccrs.epsg(3978))

# Set the extent using the appropriate coordinates and CRS
ax.set_extent([ros_snow_ice_10_slice_reprojected['x'].min(), ros_snow_ice_10_slice_reprojected['x'].max(), 
               ros_snow_ice_10_slice_reprojected['y'].min(), ros_snow_ice_10_slice_reprojected['y'].max()], crs=ccrs.epsg(3978))

# Plot the data
ros_snow_ice_10_slice_reprojected.plot(ax=ax)

# Add coastlines or other map features
ax.coastlines()

plt.show()

In [ ]:

# Assuming your rain-on-snow detection data is in a DataArray called `ros`
#labeled_array, num_features = label(ros_slice.data == 1)
#labeled_array, num_features = label(grp_slice.data < -2)
labeled_array, num_features = label(ros_snow_ice_slice.data == 1)


polygons = []
for i in range(1, num_features + 1):
    # Get the indices of the current labeled region
    indices = np.where(labeled_array == i)
    
    # Extract the x and y coordinates of these indices
    x_cluster = x_coords[indices]
    y_cluster = y_coords[indices]
    
    # Create a polygon from the cluster of coordinates
    if len(x_cluster) > 2 and len(y_cluster) > 2:  # Need at least 3 points to form a polygon
        poly = Polygon(zip(x_cluster, y_cluster))
        polygons.append(poly)

# Create a GeoDataFrame with the original CRS
gdf = gpd.GeoDataFrame(geometry=polygons, crs='EPSG:6931')


Test Convex hull

In [ ]:
from shapely.geometry import MultiPoint
# Assuming your rain-on-snow detection data is in a DataArray called `ros`
#labeled_array, num_features = label(ros_slice.data == 1)
#labeled_array, num_features = label(grp_slice.data < -2)
labeled_array, num_features = label(ros_snow_ice_slice.data == 1)

polygons = []
for i in range(1, num_features + 1):
    # Get the indices of the current labeled region
    indices = np.where(labeled_array == i)
    
    # Extract the x and y coordinates of these indices
    x_cluster = x_coords[indices]
    y_cluster = y_coords[indices]
    
    # Create a MultiPoint object from the cluster of coordinates
    if len(x_cluster) > 2 and len(y_cluster) > 2:  # Need at least 3 points to form a convex hull
        points = MultiPoint(list(zip(x_cluster, y_cluster)))
        poly = points.convex_hull  # Generate the convex hull
        polygons.append(poly)

# Create a GeoDataFrame with the original CRS
gdf = gpd.GeoDataFrame(geometry=polygons, crs='EPSG:6931')

In [ ]:
gdf_reprojected = gdf.to_crs(epsg=4326)

# Convert GeoDataFrame to GeoJSON
geojson = gdf_reprojected.to_json()

# Créer une carte centrée sur des coordonnées spécifiques
m = folium.Map(location=[75, -99], zoom_start=5)

# Add polygons to Folium map
folium.GeoJson(
    geojson,
    name='Polygons',
    style_function=lambda x: {
        'fillColor': '#FF0000',
        'color': '#FF0000',
        'weight': 1
    }
).add_to(m)

# Afficher la carte dans un Jupyter Notebook
m